<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week3_instructor_management_case.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 – Instructor demo: are bigger firms better managed?

**For live use while delivering the Week 3 online session, not a student handout.** Follows the recording slide by slide, computing every number live rather than quoting the textbook: conditional probabilities, conditional means, SDs, covariance and correlation.

Data: Békés & Kézdi's Ch. 4 case study – World Management Survey, Mexico, 2013 wave, firms with 100–5,000 employees (N = 300).

Each section header matches a slide title in the deck.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/wms_mexico_2013.csv"
firms = pd.read_csv(data_path)

print(firms.shape)
firms.head()

**Columns:** `management` = average of 18 practice scores (1–5); `emp_firm` = number of employees; `lean1` = the lean-management practice score on its own (1–5); `sic` = 2-digit industry code.

## First, each variable on its own

In [ ]:
firms[['management', 'emp_firm']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(firms['management'], bins=20, color='#457b9d', edgecolor='white')
axes[0].set(title='Management score', xlabel='Score (1–5)', ylabel='Firms')
axes[1].hist(firms['emp_firm'], bins=20, color='#2d6a4f', edgecolor='white')
axes[1].set(title='Firm size', xlabel='Employees', ylabel='Firms')
plt.tight_layout(); plt.show()

In [ ]:
# Talking point: mean vs median shows the skew
print(f"Management: mean {firms['management'].mean():.2f}, median {firms['management'].median():.2f}")
print(f"Employees:  mean {firms['emp_firm'].mean():.0f}, median {firms['emp_firm'].median():.0f}  <- long right tail")

## Conditional probabilities

Three size groups, exactly as in the textbook: small (100–199), medium (200–999), large (1,000+).

In [ ]:
firms['size'] = pd.cut(firms['emp_firm'], [0, 199, 999, np.inf],
                       labels=['Small', 'Medium', 'Large'])
firms['size'].value_counts().sort_index()

In [ ]:
# Joint distribution: counts of each (size, lean score) combination
pd.crosstab(firms['size'], firms['lean1'], margins=True)

In [ ]:
# Conditional distribution: P(lean score | size) -- each row sums to 1
cond = pd.crosstab(firms['size'], firms['lean1'], normalize='index')
cond.round(2)

In [ ]:
cond.plot(kind='barh', stacked=True, figsize=(10, 3.5), colormap='RdYlBu')
plt.xlabel('Share of firms in size group'); plt.ylabel('')
plt.legend(title='Lean score', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# The two numbers the slide asks students to read off
print(f"P(score = 1 | Small) = {cond.loc['Small', 1]:.2f}")
print(f"P(score = 5 | Large) = {cond.loc['Large', 5]:.2f}")

In [ ]:
# Q&A question 1: what would the table look like if lean score and size were independent?
# Every row would equal the overall (marginal) distribution of the lean score:
overall = firms['lean1'].value_counts(normalize=True).sort_index()
pd.DataFrame([overall] * 3, index=['Small', 'Medium', 'Large']).round(2)

## Conditional means

$E[\text{management} \mid \text{size}]$

In [ ]:
firms.groupby('size', observed=True)['management'].agg(['mean', 'count']).round(2)

## Scatter and bin-scatter

Bin-scatter: 10 bins with equal numbers of firms (`pd.qcut`), mean management score in each, plotted at the bin's median employment.

In [ ]:
firms['bin10'] = pd.qcut(firms['emp_firm'], 10, labels=False, duplicates='drop')
bins10 = firms.groupby('bin10').agg(emp=('emp_firm', 'median'),
                                    management=('management', 'mean'),
                                    n=('management', 'size'))
bins10.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(firms['emp_firm'], firms['management'], s=12, alpha=0.5, color='#457b9d')
axes[0].set(title='Scatterplot', xlabel='Employees', ylabel='Management score', ylim=(1, 5))
axes[1].plot(bins10['emp'], bins10['management'], 'o-', color='#e63946', markersize=8)
axes[1].set(title='Bin-scatter (10 equal-count bins)', xlabel='Employees (bin median)')
plt.tight_layout(); plt.show()

## A detour: log the skewed variable

In [ ]:
firms['ln_emp'] = np.log(firms['emp_firm'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(firms['emp_firm'], firms['management'], s=12, alpha=0.5, color='#457b9d')
axes[0].set(title='Employees', xlabel='Employees', ylabel='Management score')
axes[1].scatter(firms['ln_emp'], firms['management'], s=12, alpha=0.5, color='#2d6a4f')
axes[1].set(title='ln(employees)', xlabel='ln(employees)')
plt.tight_layout(); plt.show()

## Conditional distributions: box plots by size

In [ ]:
groups = [firms.loc[firms['size'] == s, 'management'] for s in ['Small', 'Medium', 'Large']]
plt.figure(figsize=(9, 3.8))
plt.boxplot(groups, vert=False)
plt.yticks([1, 2, 3], ['Small', 'Medium', 'Large'])
plt.xlabel('Management score'); plt.tight_layout(); plt.show()

In [ ]:
# The numbers behind the boxes: median and quartiles shift right, spread stays similar
firms.groupby('size', observed=True)['management'].describe()[['min', '25%', '50%', '75%', 'max', 'std']].round(2)

## Covariance and correlation

Calculate the three ingredients live, then put them together. Note: pandas divides by $n-1$ (as in the workshop); Gabor's slides divide by $n$, so his numbers can differ in the last digit.

In [ ]:
x = firms['emp_firm']
y = firms['management']

cov = ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)
sd_x = x.std()
sd_y = y.std()

print(f"Cov[employees, management] = {cov:.1f}")
print(f"SD[employees]              = {sd_x:.1f}")
print(f"SD[management]             = {sd_y:.2f}")

In [ ]:
corr = cov / (sd_x * sd_y)
print(f"Corr = {cov:.1f} / ({sd_x:.1f} x {sd_y:.2f}) = {corr:.2f}")

In [ ]:
# Same answers in one line each -- check they agree with the by-hand version
print(firms[['emp_firm', 'management']].cov().round(2))
print()
print(f"pandas .corr(): {x.corr(y):.2f}")

## Extras for the Q&A

Not on the slides – useful if questions go this way.

In [ ]:
# Correlation only captures the LINEAR part. With log employment the relationship is
# closer to a straight line (the curve in the bin-scatter flattens out), so r rises a little.
print(f"Corr[employees,      management] = {x.corr(y):.2f}")
print(f"Corr[ln(employees),  management] = {firms['ln_emp'].corr(y):.2f}")

In [ ]:
# Correlation between size and management within each industry (2-digit SIC),
# for industries with at least 15 firms -- echoes Gabor's by-industry table.
by_ind = (firms.groupby('sic')
               .apply(lambda g: pd.Series({'n': len(g), 'corr': g['emp_firm'].corr(g['management'])}),
                      include_groups=False))
by_ind[by_ind['n'] >= 15].sort_values('corr').round(2)

In [ ]:
# Does the pattern hold for each of the 18 practices on their own? Example: lean management.
print(f"Corr[employees, lean1] = {x.corr(firms['lean1']):.2f}")
firms.groupby('size', observed=True)['lean1'].mean().round(2)